# Lab 2: Scalable Pipelines for Unstructured Data

**Session 1 · Lab 2 of 3 · ~40 minutes**

## Goal

Experience the full ingestion-to-insight pipeline for unstructured financial documents.
You'll work directly with Databricks AI functions to parse, extract, and classify PDFs —
then see how those same functions are automated at scale in the Spark Declarative Pipeline.

## What you'll do

1. Explore the financial document corpus in the shared Volume
2. Upload one document to your personal Volume (hands-on UI step)
3. Parse a PDF with `ai_parse_document`
4. Extract structured fields with `ai_extract`
5. Classify AI investment signals with `ai_classify`
6. Explore the pre-built pipeline's output tables

---

In [0]:
%run ../utils/config

## Part 1: Explore the raw document corpus

The instructor has pre-staged all financial documents in the shared Volume.
Let's see what's available.

In [0]:
# List the document corpus by folder
import builtins

all_files = []
for docs_dir in dbutils.fs.ls(shared_volume):
    if docs_dir.isDir():
        folder = docs_dir.name.rstrip('/')
        pdfs = [f for f in dbutils.fs.ls(docs_dir.path) if f.name.endswith('.pdf')]
        all_files.extend([(folder, f.name, f.size) for f in pdfs])
        print(f"  {folder:<20} {len(pdfs):>3} PDFs  ({builtins.sum(f.size for f in pdfs) / 1e6:.1f} MB)")

print(f"\n  Total: {len(all_files)} PDFs")

## Part 2: Upload a sample document

**Hands-on UI step (~5 minutes)**

The instructor has placed 2 sample PDFs in the shared `financial_docs_sample` Volume.
Your task: upload one to your personal Volume using the Catalog Explorer.

**Steps:**
1. Navigate to **Catalog** → `{catalog}` → `{shared_schema}` → `financial_docs_sample`
2. Click the **Files** tab — you'll see 2 sample PDFs
3. Download one PDF to your laptop
4. Navigate to `{shared_schema}` → *your personal schema* → `financial_docs`
5. Click **Upload to this Volume** and upload the PDF

Then run the cell below to verify:

In [0]:
my_files = dbutils.fs.ls(f"/Volumes/{catalog}/{schema}/financial_docs")
pdfs = [f for f in my_files if f.name.endswith('.pdf')]

if pdfs:
    print(f"✓ Upload successful! Found {len(pdfs)} PDF(s) in your Volume:")
    for f in pdfs:
        print(f"  {f.name}  ({f.size / 1e6:.1f} MB)")
else:
    print("⚠️  No PDFs found yet. Complete the upload steps above and re-run this cell.")

## Part 3: Parse a document with `ai_parse_document`

The `ai_parse_document` function reads binary PDF content and returns structured text,
preserving headers, sections, and table content.  This is the core of ingesting PDF documents into the Bronze layer.

We'll work with one document from your personal volume

In [0]:
from pyspark.sql.functions import *

my_file = f"/Volumes/{catalog}/{schema}/financial_docs/{pdfs[0].name}"

# Read the PDF file into a dataframe while adding the "parsed" column using ai_parse_document.
df_bronze = spark.read.format("binaryFile") \
  .load(my_file) \
  .withColumnRenamed("path", "source_path") \
  .withColumnRenamed("length", "file_size_bytes") \
  .withColumn("ingested_at", current_timestamp()) \
  .drop("modificationTime")

# Write the dataframe to a Delta table in your schema
df_bronze.write.format("delta").mode("overwrite").saveAsTable("my_bronze")
display(spark.read.table("my_bronze"))

## Part 4: Extract structured data with `ai_extract`

`ai_extract` applies a schema to unstructured text and returns structured fields.
This is the core of the silver layer transformation.

In [0]:
df_silver = spark.read.table("my_bronze")

# Step 1: Parse PDF bytes with ai_parse_document().
# It returns a VARIANT containing a structured document tree.
# We extract plain text directly using :document:elements path syntax —
# this is exactly what the pipeline's silver_extract.py does.
df_silver = df_silver.withColumn(
    "plain_text",
    expr("""
        concat_ws('\n\n',
            transform(
                try_cast(ai_parse_document(content):document:elements AS ARRAY<VARIANT>),
                el -> try_cast(el:content AS STRING)
            )
        )
    """),
)

# Step 2: Extract structured fields from the plain text.
df_silver = df_silver.withColumn(
    "extracted",
    expr(
        "ai_extract(plain_text, array("
        "  'company',"
        "  'fiscal_period',"
        "  'document_type',"
        "  'revenue',"
        "  'net_income'"
        "))"
    ),
)

# Step 3: Select and rename final columns.
df_silver = df_silver.select(
    "source_path",
    "file_size_bytes",
    "ingested_at",
    "plain_text",
    col("extracted.company").alias("company"),
    col("extracted.fiscal_period").alias("fiscal_period"),
    col("extracted.document_type").alias("document_type"),
    col("extracted.revenue").alias("revenue_reported"),
    col("extracted.net_income").alias("net_income_reported"),
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("my_silver")
display(spark.read.table("my_silver"))

In [0]:
# Preview the first 2000 characters of the extracted plain text
plain_text = spark.read.table("my_silver").first().plain_text
print(plain_text[:2000])

In [0]:
# How long is the plain text? How does it compare to the raw file size?
print(f"Plain text length: {len(plain_text):,} characters")
print(f"Approximate tokens: {len(plain_text) // 4:,}")

### ai_parse_document output
`ai_parse_document()` returns a richer VARIANT with element types, page numbers, and bounding boxes — we just don't need that structure in this application, so we have immediately converted it to text using `transform()`.

For those that are curious, runn the following cell

In [0]:
display(spark.sql(
    "SELECT ai_parse_document("
        "content,"
        "map("
        "    'version', '2.0',"
        f"   'imageOutputPath', '/Volumes/{catalog}/{schema}/financial_docs/parsed_images',"
        "    'descriptionElementTypes', '*'"
        ")"
    ")"
    "AS doc_variant FROM my_bronze LIMIT 1"))

## Part 5: Classify AI investment signals with `ai_classify`

`ai_classify` assigns one of a predefined set of labels to each document.
This produces the analytical dimension that powers our gold table.

Note that Databricks has a builtin `ai_analyze_sentiment()` function which returns 'positive', 'negative', 'neutral', or 'mixed'.  But `ai_classify()` can be used as an alternative if you want more control over the labels.

In [0]:
# Classify your document — see what categories and sentiments it contains
display(spark.sql(f"""
    SELECT
        company,
        fiscal_period,
        ai_classify(
            plain_text,
            ARRAY('AI Infrastructure', 'AI Products & Services', 'AI Partnerships', 'Not AI-related')
        ) AS ai_investment_category,
        ai_analyze_sentiment(
            plain_text
        ) AS builtin_sentiment,
        ai_classify(
            plain_text,
            ARRAY('Positive', 'Neutral', 'Cautious', 'Negative')
        ) AS management_sentiment
    FROM my_silver
"""))

In [0]:
# What's the AI investment category distribution across all 5 companies in the shared gold table?
display(spark.sql(f"""
    SELECT
        company,
        ai_investment_category,
        COUNT(*) AS document_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY company), 1) AS pct_of_company
    FROM {catalog}.{shared_schema}.financial_docs_gold
    WHERE company IS NOT NULL
    GROUP BY company, ai_investment_category
    ORDER BY company, document_count DESC
"""))

## Part 6: Observe the pre-built pipeline

**In the UI:**
1. Navigate to **Workflows** → **Delta Live Tables** (or Lakeflow Pipelines)
2. Open `financial_intelligence_pipeline`
3. Click **Graph** view — observe the bronze → silver → gold → summary DAG
4. Click on any node to see its definition, expectations, and row counts

The code for this pipeline is in `../bundles/financial_pipelines/src/`

In [0]:
# The gold table is what analysts would query in practice
# What are the top AI investment insights across all companies?
display(spark.sql(f"""
    SELECT
        company,
        fiscal_period,
        ai_investment_category,
        management_sentiment,
        document_count
    FROM {catalog}.{shared_schema}.company_ai_investment_summary
    WHERE company IS NOT NULL
    ORDER BY company, fiscal_period DESC, document_count DESC
"""))

In [0]:
# Which company has the most AI Infrastructure-focused documents?
display(spark.sql(f"""
    SELECT
        company,
        SUM(document_count) AS ai_infrastructure_docs
    FROM {catalog}.{shared_schema}.company_ai_investment_summary
    WHERE ai_investment_category = 'AI Infrastructure'
    GROUP BY company
    ORDER BY ai_infrastructure_docs DESC
"""))

## Key Takeaways

1. **`ai_parse_document`** converts binary PDFs to structured text — preserving headers, sections, and tables
2. **`ai_extract`** applies a schema to unstructured text — producing typed, queryable fields without manual parsing
3. **`ai_classify`** adds analytical dimensions at SQL scale — turning documents into data warehouse rows
4. **Spark Declarative Pipelines** automate this at scale with data quality enforcement and lineage tracking
5. The same pipeline processes 130+ documents that would take an analyst days to read manually

---

**Next:** Lab 3 — Codifying data business processes

→ Open `04_codifying_processes`